---
##   Import Library

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import re
import os
import warnings
from collections import Counter

warnings.filterwarnings('ignore')
%matplotlib inline
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.family'] = 'DejaVu Sans'

RECIPES_PATH = 'clean_recipes_800_with_score.csv'
OUTPUT_PATH  = 'ingredients_master.csv'

print('Library siap!')

---
##  Load Dataset

In [ ]:
df_recipes = pd.read_csv(RECIPES_PATH)

print(f' Dataset di-load: {len(df_recipes):,} resep, {df_recipes.shape[1]} kolom')
print(f'   Distribusi kategori: {dict(df_recipes["Category"].value_counts())}')
print()
print('🔍 Sample 3 baris kolom "Ingredients Cleaned":')
print('-' * 65)
for i, val in enumerate(df_recipes['Ingredients Cleaned'].head(3), 1):
    print(f'[{i}] {str(val)[:130]}')
    print()

---
##  List Noise

In [ ]:
INSTRUKSI = {
    # cara potong
    'iris', 'diiris', 'mengiris',
    'cincang', 'dicincang', 'mencincang',
    'geprek', 'digeprek', 'memgeprek',
    'rajang', 'dirajang',
    'belah', 'dibelah', 'membelah',
    'potong', 'dipotong', 'memotong',
    'serut', 'diserut',
    'parut', 'diparut', 'memarut',
    'tumbuk', 'ditumbuk', 'menumbuk',
    'haluskan', 'dihaluskan', 'menghaluskan',
    'hancurkan', 'dihancurkan', 'menghancurkan',
    'kupas', 'dikupas', 'mengupas',
    'sisir', 'disisir',
    'serong', 'menyerong',
    'kotak', 'dadu', 'bulat', 'memanjang',
    # cara masak
    'goreng', 'digoreng', 'menggoreng',
    'rebus', 'direbus', 'merebus',
    'kukus', 'dikukus', 'mengukus',
    'bakar', 'dibakar', 'membakar',
    'tumis', 'ditumis', 'menumis',
    'oseng', 'dioseng',
    'sangrai', 'disangrai',
    'panggang', 'dipanggang',
    'ungkep', 'diungkep',
    'semur', 'disemur',
    # cara menyiapkan
    'tiriskan', 'didiamkan', 'rendam', 'direndam',
    'memarkan', 'dimemarkan',
    'lepas', 'dilepas',
    'kocok', 'dikocok',
    'campur', 'dicampur',
    'aduk', 'diaduk',
    'didihkan', 'panaskan',
    'ambil', 'buang',
    'sisihkan', 'tiriskan',
}

SATUAN = {
    # berat
    'gram', 'gr', 'g', 'kg', 'ons',
    # volume
    'ml', 'ltr', 'liter', 'cc', 'mililiter',
    # sendok
    'sdm', 'sdt', 'sendok', 'makan', 'teh',
    # wadah
    'cup', 'gelas', 'mangkok', 'piring', 'cangkir',
    # satuan bahan khusus
    'siung', 'ruas', 'bonggol', 'jempol',
    'butir', 'biji', 'buah', 'bks', 'bungkus',
    'lembar', 'helai', 'lebar',
    'batang', 'tangkai', 'ikat', 'genggam',
    'papan', 'papan', 'pcs', 'ekor', 'potong',
    'sachet', 'kaleng', 'pack', 'porsi',
    'takar', 'takaran',
    # singkatan khas cookpad
    'btg', 'lbr', 'btr', 'bh',
}


KATA_TAMBAHAN = {
    # keterangan sifat/kondisi
    'halus', 'kasar', 'tipis', 'tebal',
    'kecil', 'besar', 'sedang', 'lembut',
    'kering', 'segar', 'beku', 'matang', 'mentah', 'bersih',
    'panas', 'hangat', 'dingin', 'asin', 'manis', 'pedas',
    'dingin', 'lembek',
    # keterangan jumlah/perkiraan
    'secukupnya', 'sesuai', 'selera', 'bbrp', 'beberapa',
    'sedikit', 'banyak', 'cukup', 'kurleb', 'kira',
    'optional', 'tambahan', 'ready', 'stok', 'mumpung',
    # header section resep (artefak Cookpad)
    'bahan', 'isian', 'pelengkap', 'kuah', 'taburan',
    'topping', 'cocolan', 'marinasi', 'bumbu', 'sambal',
    # kata penghubung / keterangan
    'untuk', 'utk', 'dengan', 'dan', 'atau', 'jika',
    'kalau', 'supaya', 'agar', 'lalu', 'kemudian', 'hingga',
    'sampai', 'sebagai', 'dari', 'pada',
    # noise spesifik yang ditemukan di scan dataset ini
    'cmx', 'merkedo', 'kingfisher',   # artefak brand/ukuran
    'isinya', 'bijinya', 'minyaknya', 'airnya',  # suffix -nya
    'dihaluskan', 'rebusan', 'didihkan',
    'jk', 'msg',                       # singkatan
    'vy',                              # typo/noise
}


ALL_STOPWORDS = INSTRUKSI | SATUAN | KATA_TAMBAHAN

print('✅ Stopword dictionary siap!')
print(f'   Instruksi masak  : {len(INSTRUKSI):>4} kata')
print(f'   Satuan ukuran    : {len(SATUAN):>4} kata')
print(f'   Kata tambahan    : {len(KATA_TAMBAHAN):>4} kata')
print(f'   TOTAL stopwords  : {len(ALL_STOPWORDS):>4} kata unik')

In [ ]:
def bersihkan_bahan(teks: str) -> str:
    
    # ── Lapis 1: Regex
    # Hapus pecahan dulu (1/2, 3/4) sebelum hapus angka,
    # karena regex \d+ tidak menangkap '/' di antara angka
    teks = re.sub(r'\d+\/\d+', ' ', teks)    # pecahan: 1/2, 3/4
    teks = re.sub(r'\d+', ' ', teks)          # angka bulat
    teks = re.sub(r'[^\w\s]', ' ', teks)      # simbol: (), [], +, -, &, dll
    teks = re.sub(r'\b[a-z]\b', ' ', teks)    # huruf tunggal sisa ('x', 'g')

    # ── Lapis 2: Lowercase + strip 
    teks = teks.lower().strip()

    # ── Lapis 3: Stopword filter 
    kata_list   = teks.split()
    kata_bersih = [k for k in kata_list if k not in ALL_STOPWORDS and len(k) >= 3]

    # ── Lapis 4: Suffix cleaner 
    # Tangani varian kata kerja yang mungkin lolos dari stopword list:
    #   - Prefix 'di-'  : 'dihaluskan', 'dicincang', 'dipotong'
    #   - Prefix 'me-'  : 'menggoreng', 'memarkan', 'menumis'
    #   - Suffix '-kan' : 'haluskan', 'gorengkan', 'tiriskan'
    #   - Suffix '-i'   : 'garami', 'lumuri'
    kata_final = []
    for k in kata_bersih:
        # Skip kalau kata dimulai dengan prefix instruksi
        if re.match(r'^(di|me|mem|men|meng|ter)', k) and len(k) > 5:
            # Kecuali kata bahan yang memang berawalan 'di' atau 'me'
            # seperti 'merica', 'mentega', 'mentimun', 'daun'
            PENGECUALIAN_PREFIX = {
                'merica', 'mentega', 'mentimun', 'mentah',
                'terasi', 'tempe', 'terigu', 'tepung'
            }
            if k in PENGECUALIAN_PREFIX:
                kata_final.append(k)
            # else: buang (ini kata kerja yang lolos)
        # Skip kalau kata berakhiran '-kan' dan cukup panjang (kemungkinan kata kerja)
        elif k.endswith('kan') and len(k) > 6:
            pass  # buang: 'haluskan', 'gorengkan', 'tiriskan'
        else:
            kata_final.append(k)

    # ── Lapis 5: Validasi 
    hasil = ' '.join(kata_final)
    hasil = re.sub(r'\s+', ' ', hasil).strip()  # normalisasi spasi ganda


    if len(hasil) < 3:
        return ''

    return hasil



# Unit test 
test_cases = [
    # (input, output yang diharapkan)
    ('tahu kuning hancurkan',                  'tahu kuning'),
    ('daging sapi',                            'daging sapi'),
    ('bawang putih digoreng ambil minyaknya',  'bawang putih'),
    ('papan tempe potong',                     'tempe'),
    ('batang daun seledri',                    'daun seledri'),
    ('es batu cmx',                            'batu'),          
    ('bumbu dihaluskan',                       ''),              
    ('tiriskan',                               ''),              
    ('1/2 kg wortel iris tipis',               'wortel'),
    ('3 sdm tepung terigu',                    'tepung terigu'),
    ('margarine menumis',                      ''),              
    ('merica',                                 'merica'),        
    ('terasi',                                 'terasi'),        
    ('cabe merah kriting cabe rawit',          'cabe merah cabe rawit'),
    ('daging sapi potong dadu',                'daging sapi'),
    ('bawang putih haluskan',                  'bawang putih'), 
    ('sosis potong potongan belah',            'sosis'),
    ('jempol lengkuas muda',                   'lengkuas muda'),
    ('secukupnya air',                         'air'),
    ('buang biji',                             ''),              
]

print(' Unit Test Fungsi bersihkan_bahan():')
print('─' * 72)
print(f'{"INPUT":<45} {"OUTPUT":<25} {"STATUS"}')
print('─' * 72)

lulus = 0
for inp, expected in test_cases:
    hasil = bersihkan_bahan(inp)
    ok    = hasil == expected
    lulus += ok
    icon  = '✅' if ok else '⚠️ '
    print(f'{icon} {repr(inp):<43} → {repr(hasil):<25}', end='')
    if not ok:
        print(f'  (expected: {repr(expected)})', end='')
    print()

print('─' * 72)
print(f'   Lulus: {lulus}/{len(test_cases)} test cases')

---
## Ekstraksi & Hitung Frekuensi



In [ ]:
counter_bahan       = Counter()
total_token_mentah  = 0
total_token_dibuang = 0
log_dibuang         = Counter()  # untuk audit: noise apa yang paling sering dibuang

for baris in df_recipes['Ingredients Cleaned'].dropna():
    # Split by koma — separator yang digunakan di dataset ini
    token_list = [t.strip() for t in str(baris).split(',') if t.strip()]
    total_token_mentah += len(token_list)

    for token_raw in token_list:
        token_bersih = bersihkan_bahan(token_raw)

        if not token_bersih:
            total_token_dibuang += 1
            # Catat token asli untuk audit
            log_dibuang[token_raw.strip().lower()] += 1
            continue

        counter_bahan[token_bersih] += 1


In [ ]:
# ── Visualisasi Top 30 ────────────────────────────────────────────────────────

top30 = pd.DataFrame(
    counter_bahan.most_common(30),
    columns=['nama_bahan', 'frekuensi']
)

fig, ax = plt.subplots(figsize=(11, 8))
colors  = plt.cm.Greens(np.linspace(0.3, 0.85, 30))[::-1]
bars    = ax.barh(top30['nama_bahan'], top30['frekuensi'],
                  color=colors, edgecolor='white', linewidth=0.4)

for bar, val in zip(bars, top30['frekuensi']):
    ax.text(bar.get_width() + 1.5, bar.get_y() + bar.get_height() / 2,
            f'{val}x', va='center', fontsize=8.5, color='#444')

# Garis batas 50x — bahan di atas ini = prioritas isi umur_kulkas
ax.axvline(50, color='#E53935', linestyle='--', linewidth=1.3, alpha=0.75)
ax.text(52, 28.3, 'prioritas MVP\n(≥ 50x)', color='#E53935', fontsize=8)

ax.invert_yaxis()
ax.set_title(
    'Top 30 Bahan Paling Sering Muncul di 800 Resep\n'
    '(Setelah Aggressive Cleaning — instruksi masak sudah dibuang)',
    fontsize=13, fontweight='bold', pad=12
)
ax.set_xlabel('Frekuensi kemunculan (dari 800 resep)', fontsize=11)
ax.set_xlim(0, top30['frekuensi'].max() * 1.14)
plt.tight_layout()
plt.savefig('top30_bahan_agresif.png', bbox_inches='tight')
plt.show()
print('💾 Grafik tersimpan → top30_bahan_agresif.png')

In [ ]:
# ── Buat df_master dari hasil Counter ─────────────────────────────────────────
# Urut dari frekuensi tertinggi → bahan paling penting ada di baris teratas
# supaya waktu dibuka di Google Sheets, yang perlu diisi duluan langsung kelihatan

df_master = pd.DataFrame(
    counter_bahan.most_common(),
    columns=['nama_id', 'frekuensi']
).reset_index(drop=True)

df_master['kategori']        = 'TBD'
df_master['umur_kulkas']     = np.nan  
df_master['umur_suhu_ruang'] = np.nan  
df_master['umur_freezer']    = np.nan  

─────────────────────────────────────
n_dup = df_master['nama_id'].duplicated().sum()
if n_dup > 0:
    print(f'  {n_dup} duplikat ditemukan, otomatis dihapus.')
    df_master = df_master.drop_duplicates(subset='nama_id', keep='first').reset_index(drop=True)

print(f' df_master siap!')
print(f'   Bahan unik    : {len(df_master):,}')
print(f'   Kolom         : {list(df_master.columns)}')
print(f'   Duplikat      : {n_dup} (sudah dibersihkan)')
print()
print(' Preview 15 bahan teratas:')
df_master.head(15)

---
## Export ke CSV


Referensi pengisian
-  TKPI Kemenkes: [gizi.kemkes.go.id](https://gizi.kemkes.go.id)
- StillTasty: [stilltasty.com](https://www.stilltasty.com)

In [ ]:
df_master.to_csv(OUTPUT_PATH, index=False, encoding='utf-8')
print(f' "{OUTPUT_PATH}" tersimpan')
print(f'   {len(df_master):,} bahan | {os.path.getsize(OUTPUT_PATH)/1024:.1f} KB')

